#Creating Dimension  Dim_time

## Overview
This notebook creates the **Dim_time** dimension table for the telecom data warehouse. The time dimension provides a comprehensive calendar table with various time attributes needed for temporal analysis and reporting.

### Purpose
* Generate a complete date range covering all events in the source data
* Provide standardized time attributes (day, month, quarter, year, week)
* Enable time-based filtering and aggregation in analytics
* Support business intelligence reporting requirements

## Step 1: Determine Date Range
Query all source tables to find the minimum and maximum dates across:
* **api_data**: event_date
* **device_logs**: event_time (converted to date)
* **server_performance_data**: log_time (converted to date)

This ensures the time dimension covers the complete date range of all source data.

In [0]:
SELECT MIN(dt) AS start_date,
       MAX(dt) AS end_date
       FROM
       (
           SELECT event_date AS dt
               FROM telecom_catalog.silver_schema.api_data
               UNION ALL

            SELECT TO_DATE(event_time)
                FROM telecom_catalog.silver_schema.device_logs
                UNION ALL

            SELECT TO_DATE(log_time)
                 FROM telecom_catalog.silver_schema.server_performance_data
);

start_date,end_date
2026-05-21,2026-06-28


## Step 2: Generate Date Sequence
Create a complete calendar by generating all dates between the start and end dates using Spark SQL's `sequence()` function with a 1-day interval.

In [0]:
%python
from pyspark.sql.functions import *

dim_time = (
    spark.sql("""
                SELECT explode(
                    sequence(
                    to_date('2026-05-21'),
                    to_date('2026-06-28'),
                    interval 1 day
                            )
                        ) AS full_date""")
                     )

In [0]:
%python
dim_time.show(5)
print(dim_time.count())

+----------+
| full_date|
+----------+
|2026-05-21|
|2026-05-22|
|2026-05-23|
|2026-05-24|
|2026-05-25|
+----------+
only showing top 5 rows
39


## Step 3: Enrich with Time Attributes
Add comprehensive time dimensions to each date:
* **time_key**: Surrogate key in YYYYMMDD format (integer)
* **day**: Day of month (1-31)
* **month**: Month number (1-12)
* **month_name**: Full month name (January, February, etc.)
* **quarter**: Quarter of year (1-4)
* **year**: Year (YYYY)
* **week_of_year**: ISO week number (1-53)
* **day_of_week**: Day name (Monday, Tuesday, etc.)
* **is_weekend**: Boolean flag for Saturday/Sunday

In [0]:
%python
dim_time = (
        dim_time
            .withColumn("time_key", date_format("full_date", "yyyyMMdd").cast("int"))
            .withColumn("day", dayofmonth("full_date"))
            .withColumn("month", month("full_date"))
            .withColumn("month_name", date_format("full_date", "MMMM"))
            .withColumn("quarter", quarter("full_date"))
            .withColumn("year", year("full_date"))
            .withColumn("week_of_year", weekofyear("full_date"))
            .withColumn("day_of_week", date_format("full_date", "EEEE"))
            .withColumn("is_weekend",dayofweek("full_date").isin([1, 7])   # Sunday=1, Saturday=7
        )
    )


## Step 4: Select Final Schema
Reorder columns for the final dimension table structure, with time_key as the primary key.

In [0]:
%python
dim_time = dim_time.select(
            "time_key",
            "full_date",
            "day",
            "month",
            "month_name",
            "quarter",
            "year",
            "week_of_year",
            "day_of_week",
            "is_weekend"
         )


## Step 5: Persist to Gold Schema
Write the dimension table to **telecom_catalog.gold_schema.dim_time** using overwrite mode to ensure the latest date range is always available.

In [0]:
%python
(
    dim_time.write
        .mode("overwrite")
        .saveAsTable("telecom_catalog.gold_schema.dim_time")
)


---
## Summary

### Dimension Table Created
**Table**: `telecom_catalog.gold_schema.dim_time`

### Schema
| Column | Type | Description |
|--------|------|-------------|
| time_key | int | Primary key (YYYYMMDD format) |
| full_date | date | Complete date value |
| day | int | Day of month (1-31) |
| month | int | Month number (1-12) |
| month_name | string | Month name |
| quarter | int | Quarter (1-4) |
| year | int | Year |
| week_of_year | int | ISO week number |
| day_of_week | string | Day name |
| is_weekend | boolean | Weekend flag |

### Usage
This dimension table can be joined with fact tables using:
* **time_key** for efficient integer joins
* **full_date** for date-based joins

### Next Steps
* Join with fact tables in the gold layer
* Use for time-based filtering and aggregation in analytics
* Reference in dashboards and reports for temporal analysis